# 05 — Transfer learning: pretrained ResNet18, RGB vs 13 spectral bands

**Objective:** Fine-tune a pretrained ResNet18 for the same 10 EuroSAT land-cover labels, using exactly the **spatial train/validation/test split and train-only band statistics from notebook 02**. Run 3 seeds for RGB and for all 13 bands. The 13-band convolution is initialized with the pretrained R/G/B filters at their corresponding channels; weights for the 10 additional channels start at zero and learn during fine-tuning. This gives RGB and 13-band comparable initial *visible-band* feature extraction; non-RGB bands are not pretrained.

**Requires:** `results/split.csv`, `results/band_stats.csv`, `data/eurosat_cache.npy` (or raw EuroSAT_MS GeoTIFFs), PyTorch, and a compatible `torchvision`. On first use **ImageNet-1K pretrained weights must download**; do not silently substitute random weights. Functions: `src/piece_of_crop/resnet.py` (`build_resnet`, `resnet_param_groups`) and the training loop in `src/piece_of_crop/train.py`, the same as for the CNN in notebook 04.

**Outputs:** `models/resnet_<input>_seed<n>.pt` (local; never commit), `results/predictions/resnet_<input>_seed<n>.csv`, `results/resnet_results.csv`, `results/resnet_history.csv`, figures in `results/figures/`.

**Resource note:** This trains **six ResNet runs**, much heavier than notebook 06. Do **not** train concurrently with notebook 04. Default input resolution is 128×128 to reduce memory/time; the original proposal envisaged 224×224. If you use 128, report the deviation; `INPUT_SIZE = 224` is available for the full proposed setting. This is *selective* fine-tuning (last residual stage and classifier), not all-layer full fine-tuning. ImageNet pretrained features and train-only EuroSAT normalization are a known domain shift. Save seed and configuration for honest comparison. Always select the checkpoint on validation loss; no test-driven hyperparameter tuning.

## 1. Setup and shared data

Identical label order, row order, preprocessing, held-out test and buffers as notebook 02. All functions come from the installed `piece_of_crop` package. The cache is memory-mapped, not read wholesale into memory.

In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report

from piece_of_crop import (CLASS_NAMES, build_resnet, check_test_order, evaluate_model, find_project_root, fit,
                           get_cache, get_device, make_loaders, predictions_table, read_split,
                           resnet_param_groups, resize_images, set_seed)

SEEDS = (0, 1, 2)
BATCH_SIZE = 32
MAX_EPOCHS = 6
PATIENCE = 2
INPUT_SIZE = 128  # change to 224 for the originally proposed resolution; report choice
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-4
SKIP_COMPLETED = True  # resume completed configurations; do not skip incomplete ones

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
label_ids = np.arange(len(CLASS_NAMES))
assert np.array_equal(np.sort(split["label"].unique()), label_ids), "Unexpected class IDs"
assert cache.shape == (len(split), 13, 64, 64), "Cache/split mismatch"
device = get_device()

print("Project root:", project_root)
print("Device:", device, "Split:", split["split"].value_counts().to_dict())
print("Config:", "seeds", SEEDS, "batch", BATCH_SIZE, "input", INPUT_SIZE, "max epochs", MAX_EPOCHS)

## 2. Adapt ResNet18 to 3 vs 13 bands

ImageNet conv1 weights are in R/G/B order. The project stores Sentinel-2 bands as `B01, B02, B03, B04, ..., B8A`; RGB loader returns **B04, B03, B02** (indices 3, 2, 1). For the 13-band model put those three pretrained filters in the corresponding positions and initialize extra-channel weights to zero. Keep the same pretrained backbone and replace the 1000-class head with a 10-class head. This is a documented initialization choice, not equivalent to ImageNet pretraining on all 13 channels.

The function is `build_resnet` in `src/piece_of_crop/resnet.py`. The cell below checks the shapes and which layers are trained, without downloading the pretrained weights.

In [ ]:
for input_type, channels in [("rgb", 3), ("13band", 13)]:
    model_check = build_resnet(input_type, pretrained=False)
    assert model_check.conv1.in_channels == channels
    with torch.no_grad():
        assert model_check(resize_images(torch.zeros(2, channels, 64, 64), INPUT_SIZE)).shape == (2, 10)
    trainable = sorted({name.split(".")[0] for name, p in model_check.named_parameters() if p.requires_grad})
    print(f"{input_type}: trained layers {trainable}")
    del model_check
print("ResNet18 3- and 13-channel shape checks passed (without pretrained download).")

## 3. Training helpers and evaluation

Resize identically for both inputs immediately before the network. Earlier convolutional layers stay frozen. BatchNorm statistics in frozen layers are held fixed in training by setting those modules to evaluation mode. Validation/test loaders have fixed order. Save model checkpoints from the **lowest validation loss**; compute macro-F1 on the held-out test only after checkpoint selection.

These steps are in `src/piece_of_crop/train.py`: `fit(..., input_size=INPUT_SIZE, freeze_batchnorm=True)` resizes every batch and keeps the BatchNorm statistics of the pretrained backbone fixed. `resnet_param_groups` gives `layer4` (and the new `conv1` for 13 bands) the smaller learning rate `LR_BACKBONE` and the new head `LR_HEAD`.

## 4. Run the six configurations (3 seeds × 2 spectral inputs)

**Compute-intensive**: only run this cell when notebook 04 is finished and the Mac has enough free memory. Each completed seed is saved immediately, so an interrupted run can resume. The very first pretrained model creation downloads ImageNet weights. Do not use `torchvision` models with `weights=None` for the actual experiment. Keep `SKIP_COMPLETED=True` unless you deliberately want to rerun a seed with new settings; if settings change, remove/archive old results and checkpoints first to prevent mixing experiments.

In [ ]:
history_path = results_dir / 'resnet_history.csv'
metrics_path = results_dir / 'resnet_results.csv'

if SKIP_COMPLETED:
    saved_history = pd.read_csv(history_path).to_dict('records') if history_path.exists() else []
    saved_metrics = pd.read_csv(metrics_path).to_dict('records') if metrics_path.exists() else []
else:
    saved_history, saved_metrics = [], []

config_tag = f'resnet18_size{INPUT_SIZE}_ep{MAX_EPOCHS}_bs{BATCH_SIZE}_lr{LR_BACKBONE}_{LR_HEAD}'
for input_type in ('rgb', '13band'):
    for seed in SEEDS:
        weight_path = models_dir / f'resnet_{input_type}_seed{seed}.pt'
        prediction_path = predictions_dir / f'resnet_{input_type}_seed{seed}.csv'
        prior_rows = [r for r in saved_metrics if r.get('input') == input_type and int(r.get('seed', -1)) == seed]
        completed = (weight_path.exists() and prediction_path.exists() and
                     {r.get('split') for r in prior_rows} == {'val', 'test'} and
                     all(r.get('config') == config_tag for r in prior_rows))
        if completed and SKIP_COMPLETED:
            print(f'Skipping completed {input_type}, seed {seed}')
            continue

        print(f'\n=== ResNet18 {input_type} / seed {seed} ===', flush=True)
        saved_history = [r for r in saved_history if not (r.get('input')==input_type and int(r.get('seed',-1))==seed)]
        saved_metrics = [r for r in saved_metrics if not (r.get('input')==input_type and int(r.get('seed',-1))==seed)]
        set_seed(seed)
        loaders = make_loaders(split, cache, band_stats, input_type,
                               batch_size=BATCH_SIZE, seed=seed)
        try:
            net = build_resnet(input_type, pretrained=True).to(device)
        except Exception as exc:
            raise RuntimeError('Pretrained ResNet18 weights could not be loaded/downloaded. '
                               'Check internet access and torchvision installation; '
                               'do not continue with randomly initialized weights.') from exc
        optimizer = torch.optim.AdamW(resnet_param_groups(net, input_type, LR_BACKBONE, LR_HEAD),
                                      weight_decay=WEIGHT_DECAY)
        history, loss_fn = fit(net, loaders, optimizer, device, MAX_EPOCHS, PATIENCE,
                               run_info={"model": "ResNet18", "input": input_type, "seed": seed},
                               input_size=INPUT_SIZE, freeze_batchnorm=True)

        for part in ('val', 'test'):
            outcome = evaluate_model(net, loaders[part], loss_fn, device, input_size=INPUT_SIZE)
            saved_metrics.append({'model':'ResNet18','input':input_type,'seed':seed,
                                  'split':part,'accuracy':outcome['accuracy'],
                                  'macro_f1':outcome['macro_f1'],'loss':outcome['loss'],
                                  'n_samples':len(outcome['y_true']), 'config': config_tag})
            if part == 'test':
                check_test_order(split, outcome['y_true'])
                test_files = split.loc[split['split'].eq('test'), 'file']
                predictions_table(test_files, outcome['y_true'], outcome['proba']).to_csv(prediction_path, index=False)

        torch.save({'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()},
                    'model':'ResNet18','input':input_type,'seed':seed,
                    'num_classes':len(CLASS_NAMES),'input_size':INPUT_SIZE,
                    'config':config_tag}, weight_path)
        saved_history.extend([{**r, 'config':config_tag} for r in history])
        pd.DataFrame(saved_history).to_csv(history_path, index=False)
        pd.DataFrame(saved_metrics).to_csv(metrics_path, index=False)
        print('Saved:', weight_path.name, prediction_path.name, flush=True)
        del net, loaders
        if device.type == 'mps' and hasattr(torch, 'mps'):
            torch.mps.empty_cache()

resnet_history = pd.DataFrame(saved_history)
resnet_results = pd.DataFrame(saved_metrics)
print('Configurations completed:', len(resnet_results.query("split == 'test'")), '/ 6')
display(resnet_results.sort_values(['input','seed','split']))

## 5. Plot validation learning curves and model comparison

All shown values come from completed training runs. An incomplete run does not produce a fabricated score.

In [ ]:
import matplotlib.pyplot as plt

if resnet_history.empty or resnet_results.empty:
    raise RuntimeError('No ResNet training results yet; run the training cell first.')
for input_type in ('rgb', '13band'):
    group = resnet_history.query('input == @input_type')
    if group.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11,4))
    for seed, run in group.groupby('seed'):
        axes[0].plot(run['epoch'], run['train_loss'], ls='--',label=f'train {seed}')
        axes[0].plot(run['epoch'], run['val_loss'],label=f'val {seed}')
        axes[1].plot(run['epoch'], run['val_accuracy'],label=f'val {seed}')
    axes[0].set(title=f'ResNet18 {input_type}: loss', xlabel='Epoch', ylabel='Cross-entropy')
    axes[1].set(title=f'ResNet18 {input_type}: accuracy',xlabel='Epoch',ylabel='Accuracy',ylim=(0,1))
    for ax in axes:
        ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / f'resnet_{input_type}_training.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Test mean and std across completed seeds:')
display(resnet_results.query("split == 'test'").groupby('input')
        [['accuracy','macro_f1']].agg(['mean','std']).round(4))
for input_type in ('rgb','13band'):
    path = predictions_dir / f'resnet_{input_type}_seed0.csv'
    if path.is_file():
        pred = pd.read_csv(path)
        print(f'\nPer-class test report: {input_type}, seed 0')
        print(classification_report(pred['y_true'],pred['y_pred'], labels=label_ids,
                                    target_names=list(CLASS_NAMES),zero_division=0))

## 6. Methodological limitations for the report

- The comparison is **RGB vs 13-band** with identical splits and train-derived normalization. No test set was used to tune training settings.
- Pretraining on ImageNet RGB photos advantages the RGB pathway. The 13-band conv1 has only three pretrained visible-channel filters; its extra bands are learned from EuroSAT. **This is not 13-band pretraining.**
- The default 128×128 resolution differs from the originally proposed 224×224; document the actual resolution and selective layer4+head fine-tuning.
- These data contain land-cover labels, **not crop loss, yield or insurance-claim values**. Predictions are decision support for human review, not automatic insurance decisions.
- A spatial split reduces but does not eliminate region/scene leakage, domain shift and annotation ambiguity. Three seeds indicate sensitivity to initialization but are not a full statistical guarantee.

**External implementation reference:** [Torchvision ResNet18 documentation](https://docs.pytorch.org/vision/2.0/models/generated/torchvision.models.resnet18.html) describes `resnet18` and `ResNet18_Weights.IMAGENET1K_V1`. Actual class, split and input processing are defined by the team notebooks, not by ImageNet default transforms.